# Exercise 06: k-Nearest Neighbours, Support Vector Machines and Naive Bayes

**Dataset 1 – Palmer penguins:** body measurements of 344 penguins of three species (Adelie, Chinstrap, Gentoo) observed
on three islands in the Palmer Archipelago, Antarctica (Gorman, Williams & Fraser, 2014; Horst, Hill & Gorman, 2020).
Source: [Kaggle – Palmer Archipelago (Antarctica) penguin data](https://www.kaggle.com/datasets/parulpandey/palmer-archipelago-antarctica-penguin-data).

| variable | meaning |
|---|---|
| `species` | Adelie, Chinstrap, Gentoo – **target** |
| `culmen_length_mm`, `culmen_depth_mm` | length and depth of the bill (mm) |
| `flipper_length_mm` | flipper length (mm) |
| `body_mass_g` | body mass (g) |
| `island`, `sex` | island and sex |

**Dataset 2 – SMS Spam Collection:** 5'572 real SMS messages in English, labelled as `ham` (legitimate) or `spam`
(Almeida, Gómez Hidalgo & Yamakami, 2011). Source:
[UCI](https://archive.ics.uci.edu/dataset/228/sms+spam+collection) /
[Kaggle](https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset).

**Questions:** Can the species of a penguin be determined from a few body measurements? Can spam messages be filtered
automatically based on the words they contain?

**Slides:** Part 04 «Classification models» – k-NN, Naive Bayes, SVM.

**After this exercise you can ...**
- apply k-NN, explain why features must be scaled and choose $k$ by cross-validation,
- explain the role of the margin, the support vectors, $C$ and the kernel of an SVM,
- compute a Naive Bayes classification by hand and apply Naive Bayes to text data.

## Concept

**k-nearest neighbours (k-NN).** To classify a new point $x$: compute the distance to all training points, take the $k$
nearest $\mathcal N_k(x)$ and predict the majority class. Class probability = share of the $k$ neighbours in class $c$.
Euclidean distance:

$$d(x, x') = \sqrt{\sum_{j=1}^{p}(x_j - x'_j)^2}$$

**Caution:** features with large units dominate the distance → standardise first: $z_j = (x_j - \bar x_j)/s_j$.
Small $k$ → flexible (high variance), large $k$ → smooth (high bias).

**Naive Bayes.** Bayes' theorem with the "naive" assumption that the features are independent given the class:

$$P(y \mid x_1,\dots,x_p) \;\propto\; P(y)\,\prod_{j=1}^{p} P(x_j \mid y)$$

Start from the base rate $P(y)$ and multiply one probability per feature; predict the class with the highest score.
Normalise the scores to obtain probabilities. For text, $P(\text{word} \mid y)$ is estimated from word counts
(`MultinomialNB`; with Laplace smoothing so that unseen words do not give a probability of 0).

**Support vector machine (SVM).** Finds the boundary with the **widest margin** between the classes. Only the points on
or inside the margin – the **support vectors** – determine the boundary. $C$ controls the soft margin: small $C$ → wide
margin, some errors allowed; large $C$ → few training errors, narrow margin (risk of overfitting). A **kernel** (e.g. RBF)
measures similarity in a transformed space → curved boundaries. SVMs also need scaled features.

## Libraries and settings

In [ ]:
# Libraries
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, accuracy_score,
                             classification_report)

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

## Exercise 1: k-NN and Naive Bayes by hand

**a) k-NN with and without scaling.** A new penguin has a bill length of 45 mm and a body mass of 4'000 g. Five known
penguins:

| penguin | species | bill length [mm] | body mass [g] |
|---|---|---|---|
| P1 | Adelie | 38 | 4000 |
| P2 | Chinstrap | 46 | 3700 |
| P3 | Chinstrap | 48 | 3800 |
| P4 | Adelie | 39 | 3950 |
| P5 | Gentoo | 46 | 5000 |

1. Compute the Euclidean distances on the **original scale** and classify the new penguin with $k=3$.
2. Standardise the differences with the standard deviations of the whole data set (bill length: 5.5 mm, body mass:
   800 g), i.e. use $\Delta z = \Delta x / s$, and classify again with $k=3$.
3. Which result is more plausible? Why?

**b) Naive Bayes for spam.** In a training set of 100 SMS, 20 are spam and 80 are ham. The word "free" occurs in 12 of
the 20 spam and in 4 of the 80 ham messages; the word "call" occurs in 10 spam and 16 ham messages. A new message
contains both "free" and "call". Compute the Naive Bayes scores (as on the slide «Naive Bayes classifier», using only the
two words that occur) and the probability that the message is spam.

**c)** Verify your results with Python.

In [ ]:
# a) k-NN with and without scaling
new = np.array([45, 4000])
X_known = np.array([[38, 4000], [46, 3700], [48, 3800], [39, 3950], [46, 5000]])
species = np.array(['Adelie', 'Chinstrap', 'Chinstrap', 'Adelie', 'Gentoo'])
sd = np.array([5.5, 800])

d_raw = np.sqrt(((X_known - new)**2).sum(axis=1))
d_std = np.sqrt((((X_known - new) / sd)**2).sum(axis=1))
print(pd.DataFrame({'species': species, 'distance (raw)': d_raw.round(1),
                    'distance (standardised)': d_std.round(3)}, index=['P1', 'P2', 'P3', 'P4', 'P5']))
for name, d in [('raw', d_raw), ('standardised', d_std)]:
    nn = species[np.argsort(d)[:3]]
    print(f'{name:13s}: 3 nearest = {nn.tolist()}')

# b) Naive Bayes
score_spam = 20/100 * 12/20 * 10/20
score_ham  = 80/100 * 4/80 * 16/80
print(f'\nscore(spam) = {score_spam:.4f}, score(ham) = {score_ham:.4f}')
print(f'P(spam | free, call) = {score_spam / (score_spam + score_ham):.3f}')

**Solution:**

a) 1. Raw distances: P1 = 7.0, P2 ≈ 300.0, P3 ≈ 200.0, P4 ≈ 50.4, P5 ≈ 1000.0 → 3 nearest: P1, P4, P3 →
**Adelie** (2 of 3). The distance is determined almost entirely by the body mass: 1 g counts as much as 1 mm.

2. Standardised: P1 = 7/5.5 ≈ 1.273; P2 = $\sqrt{0.182^2+0.375^2}$ ≈ 0.417; P3 = $\sqrt{0.545^2+0.25^2}$ ≈ 0.600;
P4 = $\sqrt{1.091^2+0.0625^2}$ ≈ 1.093; P5 = $\sqrt{0.182^2+1.25^2}$ ≈ 1.263 → 3 nearest: P2, P3, P4 →
**Chinstrap** (2 of 3).

3. The standardised result is more plausible: a bill of 45 mm is long for an Adelie (typically below 40 mm) but typical
for a Chinstrap. Without scaling, the bill length is practically ignored.

b) score(spam) = 0.2 · 0.6 · 0.5 = 0.06; score(ham) = 0.8 · 0.05 · 0.2 = 0.008 → predict **spam**;
P(spam | free, call) = 0.06 / (0.06 + 0.008) ≈ **0.88**. Although only 20 % of all messages are spam, the word "free"
(12 times more frequent in spam) makes spam very likely.

## Exercise 2: Explore the penguin data

a) Import the penguin data, remove rows with missing values and the row with `sex == '.'`.

b) Create a scatter plot matrix (`sns.pairplot(..., hue='species')`) of the four numerical measurements. Which pairs of
features separate the species well? Compare the scales of the features.

c) Split the data into a stratified training (80 %) and test set (20 %) with `random_state=42`, using the four numerical
measurements as features.

In [ ]:
# a) Import the data
peng = pd.read_csv('./Data/penguins.csv', sep=',')
peng = peng.dropna()
peng = peng[peng['sex'] != '.'].reset_index(drop=True)
print(peng.shape)
print(peng['species'].value_counts())

# b) Pairplot
features = ['culmen_length_mm', 'culmen_depth_mm', 'flipper_length_mm', 'body_mass_g']
print(peng[features].describe().round(1))
sns.pairplot(peng, vars=features, hue='species', height=2.2, plot_kws={'s': 15})
plt.show()

# c) Train/test split
X_train, X_test, y_train, y_test = train_test_split(peng[features], peng['species'], test_size=0.20,
                                                    stratify=peng['species'], random_state=42)
print('Training set:', X_train.shape, ' Test set:', X_test.shape)

**Solution:**

b) Gentoo penguins are clearly separated by flipper length, body mass and bill depth (large, but with a shallow bill).
Adelie and Chinstrap have similar body size, but differ in **bill length** (Chinstrap: longer bill). The combination
bill length × bill depth separates all three species well. The scales differ enormously: body mass is in the thousands
(standard deviation ≈ 800 g), bill depth varies by only a few mm (standard deviation ≈ 2 mm).

## Exercise 3: k-NN – the effect of scaling

a) Compute the 5-fold CV accuracy (on the training set) of `KNeighborsClassifier(n_neighbors=5)` on the **unscaled**
features and of `make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))`.

b) Explain the difference. Why must the scaler be part of the pipeline (and not be fitted on the whole data set
beforehand)?

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

knn_raw    = KNeighborsClassifier(n_neighbors=5)
knn_scaled = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
for name, m in [('unscaled', knn_raw), ('scaled', knn_scaled)]:
    acc = cross_val_score(m, X_train, y_train, cv=cv, scoring='accuracy')
    print(f'k-NN ({name:8s}): CV accuracy = {acc.mean():.3f} (+/- {acc.std():.3f})')

**Solution:**

Without scaling the accuracy is only about **0.76**, with scaling about **0.99**. Without scaling, the distance is
dominated by `body_mass_g` (differences of hundreds of grams vs. a few millimetres); the bill length, which separates
Adelie and Chinstrap, is practically ignored – exactly the effect from Exercise 1a.

The scaler must be fitted on the training folds only; otherwise the means and standard deviations would contain
information from the validation/test data (**data leakage**). In a pipeline, `cross_val_score` re-fits the scaler in
every fold automatically.

## Exercise 4: k-NN – choosing k

a) Compute the 5-fold CV accuracy of the scaled k-NN for $k = 1, 2, \dots, 40$ and plot it against $k$.

b) Choose $k$ (if several values are equally good, prefer the larger one – it gives a smoother model). Fit the model on
the whole training set and compute accuracy and confusion matrix on the test set.

c) Use the model to classify a new penguin with bill length 45 mm, bill depth 18 mm, flipper length 195 mm and body mass
4'000 g. Print the class probabilities (`predict_proba`).

In [ ]:
# a) CV accuracy for k = 1..40
ks = range(1, 41)
cv_acc = [cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k)),
                          X_train, y_train, cv=cv).mean() for k in ks]

plt.figure(figsize=(7, 3.5))
plt.plot(ks, cv_acc, marker='o', markersize=4)
plt.xlabel('k (number of neighbours)')
plt.ylabel('5-fold CV accuracy')
plt.grid(alpha=0.3)
plt.show()

# b) Final model and test evaluation
best_k = max(k for k, a in zip(ks, cv_acc) if a == max(cv_acc))
knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=best_k)).fit(X_train, y_train)
print(f'Chosen k = {best_k}, CV accuracy = {max(cv_acc):.3f}, test accuracy = {knn.score(X_test, y_test):.3f}')
ConfusionMatrixDisplay.from_estimator(knn, X_test, y_test, cmap='Blues', colorbar=False)
plt.title(f'k-NN (k = {best_k}) on the test set')
plt.show()

# c) Classify a new penguin
new_penguin = pd.DataFrame({'culmen_length_mm': [45], 'culmen_depth_mm': [18],
                            'flipper_length_mm': [195], 'body_mass_g': [4000]})
print(pd.DataFrame(knn.predict_proba(new_penguin), columns=knn.classes_).round(2))

**Solution:**

a) Already $k=1$ is very good on this easy problem; the CV accuracy stays at about 0.97–0.99 for a wide range of $k$
and tends to decrease for large $k$ (the boundary becomes too smooth; the small Chinstrap class is "outvoted").

b) The selected $k = 5$ classifies all penguins of the test set correctly (test accuracy 1.0). With only 67 test
penguins, this should not be over-interpreted – the CV accuracy (≈ 0.985) is the more stable estimate.

c) The new penguin is a **borderline case**: 3 of its 5 nearest neighbours are Chinstrap, 2 are Adelie →
P(Chinstrap) = 0.6, P(Adelie) = 0.4 – exactly the "share of the $k$ neighbours" from the slide. Its bill length fits
Chinstrap, its body mass and flipper length fit both.

## Exercise 5: Support vector machines – margin, C and kernel

To visualise the decision boundaries we use only the two features `culmen_length_mm` and `culmen_depth_mm`.

a) Fit `make_pipeline(StandardScaler(), SVC(kernel=..., C=...))` for `kernel` ∈ {`'linear'`, `'rbf'`} and
$C$ ∈ {0.01, 1, 100} on the training set. Plot the 6 decision boundaries in a 2 × 3 grid
(the decision regions are drawn by predicting the species on a fine grid), with the training points and the **support vectors** circled.

b) Print the number of support vectors and the 5-fold CV accuracy for each model.

c) Describe the influence of $C$ and of the kernel. Which model would you choose?

In [ ]:
features_2d = ['culmen_length_mm', 'culmen_depth_mm']
X2_train = X_train[features_2d]
colors = {'Adelie': 'tab:orange', 'Chinstrap': 'tab:purple', 'Gentoo': 'tab:green'}

# Grid of points covering the feature space (used to draw the decision regions)
xx, yy = np.meshgrid(np.linspace(32, 61, 300), np.linspace(12.5, 22.5, 300))
grid = pd.DataFrame({features_2d[0]: xx.ravel(), features_2d[1]: yy.ravel()})

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True, sharey=True)
results = []
for row, kernel in enumerate(['linear', 'rbf']):
    for col, C in enumerate([0.01, 1, 100]):
        ax = axes[row, col]
        svm = make_pipeline(StandardScaler(), SVC(kernel=kernel, C=C)).fit(X2_train, y_train)
        # Predict the species on a fine grid and colour the regions like the points
        zz = pd.Categorical(svm.predict(grid), categories=list(colors)).codes.reshape(xx.shape)
        ax.contourf(xx, yy, zz, levels=[-0.5, 0.5, 1.5, 2.5], colors=list(colors.values()), alpha=0.2)
        for sp, c in colors.items():
            m = y_train == sp
            ax.scatter(X2_train.loc[m, features_2d[0]], X2_train.loc[m, features_2d[1]],
                       s=12, color=c, label=sp)
        sv = X2_train.iloc[svm[-1].support_]
        ax.scatter(sv[features_2d[0]], sv[features_2d[1]], s=60, facecolors='none',
                   edgecolors='black', linewidths=0.8, label='support vectors')
        n_sv = len(svm[-1].support_)
        cv_acc_svm = cross_val_score(svm, X2_train, y_train, cv=cv).mean()
        results.append({'kernel': kernel, 'C': C, 'support vectors': n_sv, 'CV accuracy': round(cv_acc_svm, 3)})
        ax.set_title(f'kernel = {kernel}, C = {C}: {n_sv} support vectors')
axes[0, 0].legend(loc='lower left', fontsize=8)
for ax in axes.flat:
    ax.set_xlabel('Bill length [mm]' if ax in axes[1] else '')
    ax.set_ylabel('Bill depth [mm]' if ax in axes[:, 0] else '')
plt.show()

# b) Number of support vectors and CV accuracy
print(pd.DataFrame(results).to_string(index=False))

**Solution:**

- **Small $C$ (0.01):** very wide margin, many training errors are tolerated → most points are support vectors (194
  resp. 244 of 266) and the boundaries are too simple. With the RBF kernel, Chinstrap is never predicted and the CV
  accuracy drops to ≈ 0.49 → **underfitting**.
- **$C = 1$:** far fewer support vectors (42 resp. 54), CV accuracy ≈ 0.95 for both kernels.
- **Large $C$ (100):** narrow margin, the model tries to classify every training point correctly (22 resp. 27 support
  vectors). The linear SVM improves slightly (≈ 0.97); with the RBF kernel the boundaries become very curved and form
  "islands" around single points, and the CV accuracy decreases slightly (≈ 0.94) → **overfitting**.
- **Kernel:** the linear kernel gives straight boundaries, the RBF kernel curved ones. Here the classes are almost
  linearly separable, so a **linear SVM** with $C$ between 1 and 100 is a good, simple choice. In general, $C$ (and the
  kernel parameters) must be tuned with cross-validation.

The number of support vectors decreases with increasing $C$: only the points on or inside the (narrower) margin determine
the boundary.

## Exercise 6: Naive Bayes spam filter

a) Import the SMS data (`encoding='latin-1'`), keep only the first two columns and rename them to `label` and `text`.
How many messages are spam? Remove duplicated messages.

b) Split the data into a stratified training (80 %) and test set (20 %) with `random_state=42`.

c) Transform the texts into word counts with `CountVectorizer(stop_words='english')` (fit on the training texts only!)
and fit a `MultinomialNB()`. How many different words (features) are there?

d) Evaluate the model on the test set (confusion matrix, classification report). Which error is worse for a spam filter:
a false positive (ham classified as spam) or a false negative?

In [ ]:
# a) Import the data
sms = pd.read_csv('./Data/sms_spam.csv', encoding='latin-1')
sms = sms.iloc[:, :2]
sms.columns = ['label', 'text']
print(sms['label'].value_counts())
print('Duplicated messages:', sms.duplicated().sum())
sms = sms.drop_duplicates().reset_index(drop=True)
print('Share of spam after removing duplicates:', round((sms['label'] == 'spam').mean(), 3))
print(sms.head(5).to_string())

# b) Train/test split
txt_train, txt_test, lab_train, lab_test = train_test_split(sms['text'], sms['label'], test_size=0.20,
                                                            stratify=sms['label'], random_state=42)

# c) Bag of words and Naive Bayes
vectorizer = CountVectorizer(stop_words='english')
W_train = vectorizer.fit_transform(txt_train)
W_test  = vectorizer.transform(txt_test)
print('Number of words (features):', W_train.shape[1])

nb = MultinomialNB()
nb.fit(W_train, lab_train)

# d) Evaluation
lab_pred = nb.predict(W_test)
ConfusionMatrixDisplay.from_predictions(lab_test, lab_pred, cmap='Blues', colorbar=False)
plt.title('Naive Bayes spam filter (test set)')
plt.show()
print(classification_report(lab_test, lab_pred, digits=3))

**Solution:**

a) 747 of the 5'572 messages are spam (≈ 13 %); 403 messages are duplicates (mostly identical spam or standard replies).
After removing them, 5'169 messages remain (≈ 12.6 % spam). Duplicates must be removed, otherwise the same message
could end up in the training and in the test set.

c) The vocabulary contains about 7'300 different words – far more features than messages. Naive Bayes handles this
easily, because it only has to estimate one word frequency per class and word.

d) The model reaches an accuracy of about 0.985. For spam, precision is ≈ 0.96 (almost all messages flagged as spam
are really spam), and recall is ≈ 0.92 (some spam messages get through). For a spam filter, a **false positive** is
usually worse: an important legitimate message (e.g. from the doctor or employer) disappears in the spam folder,
whereas a spam message in the inbox is only annoying. High precision for the class spam is therefore essential.

## Exercise 7: What has the spam filter learned?

a) The attribute `nb.feature_log_prob_` contains $\log P(\text{word} \mid \text{class})$ for both classes. Compute for
each word the log ratio $\log P(\text{word} \mid \text{spam}) - \log P(\text{word} \mid \text{ham})$ and print the 15
words that indicate spam most strongly.

b) Classify the following messages and print $P(\text{spam})$. Write two messages of your own.

In [ ]:
# a) Most indicative spam words
words = vectorizer.get_feature_names_out()
spam_idx = list(nb.classes_).index('spam')
ham_idx  = list(nb.classes_).index('ham')
log_ratio = nb.feature_log_prob_[spam_idx] - nb.feature_log_prob_[ham_idx]
top_spam = pd.Series(log_ratio, index=words).sort_values(ascending=False).head(15)
print('Words that indicate spam most strongly (log ratio):')
print(top_spam.round(2).to_string())

# b) Classify new messages
new_messages = ['Congratulations! You have won a free prize. Call now to claim your reward',
                'Hi, are we still meeting for lunch tomorrow?',
                'URGENT: your account has been selected, reply YES to claim cash',
                'Can you call me when you are free?']
p_spam = nb.predict_proba(vectorizer.transform(new_messages))[:, spam_idx]
for msg, p in zip(new_messages, p_spam):
    print(f'P(spam) = {p:.3f}  |  {msg}')

**Solution:**

a) The strongest spam indicators are words like *claim*, *prize*, *150p*, *nokia*, *tone*, *www*, *guaranteed*,
*awarded*, *uk* and numbers such as *18*, *500* or *1000* – typical for prize, ringtone and subscription scams. Note that
the vectorizer treats numbers such as "150p" (price per message) as words.

b) The first and third messages get a spam probability of ≈ 1, the lunch message ≈ 0. The last message contains "call"
and "free", two words that are much more frequent in spam (as in Exercise 1b), but in a harmless context: it gets
P(spam) ≈ 0.51 and is (just) classified as **spam – a false positive**. This shows the limits of the "bag of words"
assumption: Naive Bayes ignores word order and context – and it assumes the words to be
independent, which is clearly not true ("free" and "prize" occur together). Nevertheless it works remarkably well and is
extremely fast.

## Exercise 8: Reflection

Compare k-NN, SVM and Naive Bayes: for which of the three methods is

a) feature scaling necessary,

b) the training fast but the prediction slow,

c) a very large number of features (e.g. thousands of words) unproblematic?

**Solution:**

a) **k-NN and SVM** are based on distances / inner products → scale the features. Naive Bayes estimates one distribution
per feature and class, scaling is not needed.

b) **k-NN** has practically no training ("lazy learner": it only stores the data), but for each prediction the distances
to all training points must be computed.

c) **Naive Bayes** (one parameter per word and class) and linear **SVMs** (widely used for text classification) cope well
with many features. For k-NN, distances in very high-dimensional spaces become less informative ("curse of
dimensionality").

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')